In [1]:
import pandas as pd
import os

# Define paths
processed_dir = r'A:\desktop\New folder\P\Ratul\Supply_Chain_Thesis\data\processed'

print("Loading chronological raw data to generate human-readable LLM prompts...")
# 1. Load the original cleaned data to get the UNSCALED, readable text
df_raw = pd.read_csv(os.path.join(processed_dir, 'cleaned_data.csv'))
df_raw['order_date'] = pd.to_datetime(df_raw['order_date'])

# 2. Sort it chronologically (Exactly how we did in Notebook 03 to ensure indices match)
df_raw = df_raw.sort_values('order_date').reset_index(drop=True)

# 3. Grab the test set split (the last 20%)
split_idx = int(len(df_raw) * 0.8)
test_raw = df_raw.iloc[split_idx:].head(5).copy() # Take the first 5 test records for our examples

# Framework Generator Function
def generate_supply_chain_prompt(row):
    prompt = f"""
    SYSTEM: You are an expert Supply Chain & Logistics AI. 
    TASK: Predict the delivery lead time (in days) for the following shipment based on the provided parameters.

    [SHIPMENT CONTEXT]
    - Product Category (NLP): {row.get('category_name', 'Unknown')}
    - Shipping Mode: {row.get('shipping_mode', 'Unknown')}
    - Market/Region: {row.get('market', 'Unknown')} / {row.get('order_region', 'Unknown')}
    - Order Month: {row['order_date'].month}
    - Order Quantity: {row.get('order_item_quantity', 'Unknown')}
    - Total Sales Value: ${row.get('sales', 0):.2f}

    [INSTRUCTION]
    Based on the historical volatility of this shipping mode and the physical routing constraints, estimate the lead time in days. 
    Provide your reasoning, followed by a final numerical prediction.
    
    [ACTUAL RESULT FOR THESIS EVALUATION: {row.get('days_for_shipping_real', 0):.1f} days]
    """
    return prompt

# Apply the prompt generation framework
print("\nGenerating Few-Shot Prompts...\n")
test_raw['llm_prompt'] = test_raw.apply(generate_supply_chain_prompt, axis=1)

# Display the framework output for the thesis document
for i, prompt in enumerate(test_raw['llm_prompt']):
    print(f"================ PROMPT {i+1} ================")
    print(prompt)
    print("============================================\n")

print("Prompt-Based Framework successfully generated! You can copy these outputs into the Thesis 'Methodology' section.")

Loading chronological raw data to generate human-readable LLM prompts...

Generating Few-Shot Prompts...

================ PROMPT 1 ================

    SYSTEM: You are an expert Supply Chain & Logistics AI. 
    TASK: Predict the delivery lead time (in days) for the following shipment based on the provided parameters.

    [SHIPMENT CONTEXT]
    - Product Category (NLP): Men's Footwear
    - Shipping Mode: Standard Class
    - Market/Region: LATAM / Central America
    - Order Month: 4
    - Order Quantity: 1
    - Total Sales Value: $129.99

    [INSTRUCTION]
    Based on the historical volatility of this shipping mode and the physical routing constraints, estimate the lead time in days. 
    Provide your reasoning, followed by a final numerical prediction.

    [ACTUAL RESULT FOR THESIS EVALUATION: 6.0 days]
    

================ PROMPT 2 ================

    SYSTEM: You are an expert Supply Chain & Logistics AI. 
    TASK: Predict the delivery lead time (in days) for the followi